In [1]:
import pandas as pd
import numpy as np
import pyarrow
from src_RN import *
import itertools
import gc 

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import  train_test_split
from sklearn.metrics import classification_report

import keras
from keras.models import Sequential
from keras.layers import Dense, Input, Dropout
from keras import optimizers

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.linear_model import LogisticRegression

print(tf.config.list_physical_devices())
BATCH_SIZE = 64

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando Redes Neurais

In [2]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet("enem_parquet", columns = colunas)

## 1.2- Pré-Processando os Dados

In [3]:
df = pre_processor_rn(df, objetivo = '', n_samples = 50000)

## 1.3- Construção da Matriz X e Vetor y

In [6]:
X = df.drop([ 'FALTOU'], axis=1)

y = df['FALTOU']

## 1.4 - Separação em Dados de Treino, Validação e Teste

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

## 1.5 - Tratando os Dados

In [8]:
preprocessador = transformar(X_train)

X_train = preprocessador.transform(X_train).astype(np.float32)
X_val   = preprocessador.transform(X_val).astype(np.float32)
X_test  = preprocessador.transform(X_test).astype(np.float32)

## 1.6 - Treinando a Rede Neural

In [9]:
max_neurons = num_max_neuronio(X_train, d = X_train.shape[1])
print("Número máximo de neurônios:", max_neurons)

Número máximo de neurônios: 19


In [10]:
param_grid = {
    'neurons':       [18],
    'learning_rate': [0.0001, 0.001, 0.01, 0.1],
    'batch_size':    [32,64],             
    'epochs':        [100],
    'l2_reg':        [0.001, 0.01, 0.1],          
    'dropout':       [0.0, 0.2],
}

keys, values = zip(*param_grid.items())
combinacoes = [dict(zip(keys, v)) for v in itertools.product(*values)]
print(f"Total de combinações: {len(combinacoes)}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

resultados = []

for params in combinacoes:
    print(f"Testando: {params}")
    
    accs = []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val_fold = X_train[train_idx], X_train[val_idx]
        y_tr, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = create_model(
            input_dim=X_train.shape[1],
            neurons=params['neurons'],
            learning_rate=params['learning_rate'],
            l2_reg=params['l2_reg'],
            dropout=params['dropout']
        )

        early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val_fold, y_val_fold),
            epochs=params['epochs'],
            batch_size=params['batch_size'],
            callbacks=[early_stop],
            verbose=1
        )

        loss, acc = model.evaluate(X_val_fold, y_val_fold, verbose=0)
        accs.append(acc)

        del model
        gc.collect()

    mean_acc = np.mean(accs)

    resultados.append({
        'params': params,
        'mean_accuracy': mean_acc
    })

    print(f"Accuracy média: {mean_acc:.4f}")

Total de combinações: 48
Testando: {'neurons': 18, 'learning_rate': 0.0001, 'batch_size': 32, 'epochs': 100, 'l2_reg': 0.001, 'dropout': 0.0}
Epoch 1/100
239/239 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7374 - loss: 0.6051 - val_accuracy: 0.7421 - val_loss: 0.5870
Epoch 2/100
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7373 - loss: 0.5925 - val_accuracy: 0.7411 - val_loss: 0.5797
Epoch 3/100
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7416 - loss: 0.5860 - val_accuracy: 0.7442 - val_loss: 0.5748
Epoch 4/100
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7420 - loss: 0.5810 - val_accuracy: 0.7442 - val_loss: 0.5704
Epoch 5/100
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7424 - loss: 0.5770 - val_accuracy: 0.7469 - val_loss: 0.5670
Epoch 6/100
239/239 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7438 - loss: 0.5735 - val_accuracy: 0.7474 - val_loss: 0.5639
Epoch 7/100
239/239 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7444 - loss: 0.57

### Salvando os resultados

In [13]:
resultados_df = pd.DataFrame(resultados)
resultados_df = resultados_df.sort_values(by='mean_accuracy', ascending=False)

In [16]:
resultados_df.to_csv("resultados_presenca_socioeconomicos.csv", index=False)

## 1.8 - Treinando com todos os dados de treino

In [33]:
X = df.drop(columns=['FALTOU'])
y = df['FALTOU']

In [34]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [35]:
preprocessador = transformar(X_train)

X_train = preprocessador.transform(X_train).astype(np.float32)
X_test  = preprocessador.transform(X_test).astype(np.float32)

In [36]:
BATCH_SIZE = 32

adam = Adam(learning_rate=0.01)

model = Sequential()

model.add(Input(shape=(X_train.shape[1],)))

model.add(Dense(18, kernel_initializer='he_normal', kernel_regularizer=regularizers.l2(0.001), 
                activation='relu'))

model.add(Dense(1, kernel_initializer='he_normal', activation='sigmoid'))

es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10)

# Compilar o modelo
model.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])

print(model.summary())

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 18)             │           846 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            19 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 865 (3.38 KB)

 Trainable params: 865 (3.38 KB)

 Non-trainable params: 0 (0.00 B)

None


In [37]:
history = model.fit(X_train, y_train,
                    epochs=100,
                    batch_size=BATCH_SIZE) 

# Obtendo a acuracia no conjunto de treinamento
E_in, acc_train = model.evaluate(X_train, y_train, batch_size=BATCH_SIZE, verbose=0)

# Obtendo a acuracia no conjunto de teste
E_test, acc_test = model.evaluate(X_test, y_test, batch_size=BATCH_SIZE, verbose=0)

print(f"--> E_test - E_in = {E_test - E_in:.4f}")
print(f'--> Acuracia (treino): {acc_train:.4f}')
print(f'--> Acuracia (teste): {acc_test:.4f}')
print(f"--> acc_train - acc_test = {acc_train - acc_test:.4f}")

Epoch 1/100


373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7411 - loss: 0.5705
Epoch 2/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7484 - loss: 0.5415
Epoch 3/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7508 - loss: 0.5360
Epoch 4/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7508 - loss: 0.5361
Epoch 5/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7484 - loss: 0.5343
Epoch 6/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7491 - loss: 0.5343
Epoch 7/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7479 - loss: 0.5332
Epoch 8/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7525 - loss: 0.5319
Epoch 9/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7513 - loss: 0.5324
Epoch 10/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7514 - loss: 0.5326
Epoch 11/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7516 - loss: 0.5330
Epoch 12/100
373/373 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

In [38]:
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred))

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
              precision    recall  f1-score   support

           0       0.76      0.97      0.85      2177
           1       0.65      0.15      0.25       803

    accuracy                           0.75      2980
   macro avg       0.70      0.56      0.55      2980
weighted avg       0.73      0.75      0.69      2980

